# Разметка изображений в Google Colab

Блокнот размечает все изображения из папки `data` с помощью `HuggingFaceTB/SmolVLM-256M-Instruct` и сохраняет результат в `dataset.jsonl`.

Рекомендуется запускать в Google Colab с GPU: `Runtime` -> `Change runtime type` -> `T4 GPU`.

## 1. Подготовка окружения

Загрузите папку `data` рядом с этим блокнотом в Colab или раскомментируйте ячейку с Google Drive ниже.

In [ ]:
# Colab: ставим свежие версии библиотек для SmolVLM.
# Pillow фиксируем ниже 12, чтобы не конфликтовать с gradio в Colab.
!pip -q install -U "transformers>=4.51.0" "accelerate>=0.31.0" "pillow>=8.0,<12.0" "tqdm" "safetensors"

In [ ]:
# Если данные лежат на Google Drive, раскомментируйте эти строки и укажите свою папку.
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/seminar_05

In [ ]:
import gc
import json
from pathlib import Path

import torch
from PIL import Image
from tqdm.auto import tqdm

try:
    from transformers import AutoModelForImageTextToText as AutoModelClass
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoModelClass

from transformers import AutoProcessor

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
IMAGE_DIR = Path("data")
OUTPUT_FILE = Path("dataset.jsonl")
CSV_FILE = Path("dataset.csv")

MAX_NEW_TOKENS = 128
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Device:", DEVICE)
print("Model dtype:", MODEL_DTYPE)

## 2. Проверка данных

In [ ]:
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        f"Папка {IMAGE_DIR.resolve()} не найдена. "
        "Загрузите папку data рядом с блокнотом или подключите Google Drive."
    )

image_files = sorted(
    path for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not image_files:
    raise FileNotFoundError(f"В папке {IMAGE_DIR.resolve()} нет изображений.")

print(f"Найдено изображений: {len(image_files)}")
print("Первый файл:", image_files[0].as_posix())

In [ ]:
import matplotlib.pyplot as plt

sample_files = image_files[:6]
fig, axes = plt.subplots(1, len(sample_files), figsize=(3 * len(sample_files), 3))

if len(sample_files) == 1:
    axes = [axes]

for ax, image_path in zip(axes, sample_files):
    ax.imshow(Image.open(image_path).convert("RGB"))
    ax.set_title(image_path.name[:24], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Загрузка SmolVLM

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
}

if DEVICE == "cuda":
    model_kwargs["_attn_implementation"] = "eager"

try:
    model = AutoModelClass.from_pretrained(MODEL_ID, **model_kwargs)
except Exception as exc:
    if "attn" not in str(exc).lower():
        raise
    model_kwargs.pop("_attn_implementation", None)
    model = AutoModelClass.from_pretrained(MODEL_ID, **model_kwargs)

model = model.to(DEVICE)
model.eval()

print("Модель загружена:", MODEL_ID)

## 4. Функция разметки

Для SmolVLM промпт написан на английском языке.

In [ ]:
PROMPT = """
Describe this image for a computer vision training dataset.
Mention the visible people, their clothing or protective equipment, the main objects, the action, and the scene/location.
Answer in one concise but specific English sentence.
Do not invent details that are not visible.
""".strip()


def clean_answer(text: str) -> str:
    text = text.strip()
    for prefix in ("Assistant:", "assistant:"):
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    return " ".join(text.split())


def caption_image(image_path: Path) -> str:
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=prompt,
        images=[image],
        return_tensors="pt",
    ).to(DEVICE)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    new_tokens = generated_ids[:, inputs["input_ids"].shape[-1]:]
    output_text = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return clean_answer(output_text)

In [ ]:
test_image = image_files[0]
display(Image.open(test_image).convert("RGB"))

test_caption = caption_image(test_image)
print("Image:", test_image.as_posix())
print("Caption:", test_caption)

## 5. Разметка всей папки

Ячейка умеет продолжать работу: если `dataset.jsonl` уже существует, успешно размеченные изображения не будут размечаться повторно.

In [ ]:
def read_completed_images(output_file: Path) -> set[str]:
    completed = set()

    if not output_file.exists():
        return completed

    with output_file.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue

            if item.get("image") and item.get("text"):
                completed.add(item["image"])

    return completed


completed_images = read_completed_images(OUTPUT_FILE)
images_to_process = [
    path for path in image_files
    if path.as_posix() not in completed_images
]

print(f"Уже размечено: {len(completed_images)}")
print(f"Осталось разметить: {len(images_to_process)}")

with OUTPUT_FILE.open("a", encoding="utf-8") as f:
    for image_path in tqdm(images_to_process):
        image_key = image_path.as_posix()

        try:
            caption = caption_image(image_path)
            item = {
                "image": image_key,
                "text": caption,
            }
        except Exception as exc:
            item = {
                "image": image_key,
                "text": "",
                "error": repr(exc),
            }

        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        f.flush()

        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

print("Saved to", OUTPUT_FILE.as_posix())

## 6. Проверка и экспорт

In [ ]:
import pandas as pd

rows = []
with OUTPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.to_csv(CSV_FILE, index=False)

print(f"JSONL rows: {len(df)}")
print("JSONL:", OUTPUT_FILE.as_posix())
print("CSV:", CSV_FILE.as_posix())
display(df.tail(10))

In [ ]:
try:
    from google.colab import files

    files.download(str(OUTPUT_FILE))
    files.download(str(CSV_FILE))
except Exception as exc:
    print("Скачивание через files.download доступно только в Google Colab.")
    print(exc)